In [16]:
from sequence import load_data, get_vocab
import os
import numpy as np
from time_layers import TimeEmbedding, TimeAffine, TimeSoftmaxWithLoss

In [14]:
(x_train, t_train), (x_test, t_test) = load_data('addition.txt', seed=1984)
char_to_id, id_to_char = get_vocab()

# x_train, t_train에는 문자 ID의 시퀀스가 저장되어 있다.
print(x_train.shape, t_train.shape)  # (45000, 7) (45000, 5) -> 45000개의 데이터가 있고, x_train 데이터에는 7개의 문자로 이루어져 있다.
print(x_test.shape, t_test.shape)

(45000, 7) (45000, 5)
(5000, 7) (5000, 5)


In [15]:
print(x_train[0]) # 0~9, +, ' '의 문자 ID로 이루어진 7개의 문자로 이루어진 문제
print(''.join([id_to_char[c] for c in x_train[0]])) # 각 문자 ID에 해당하는 문자를 반환, 문제
print(''.join([id_to_char[c] for c in t_train[0]])) # 각 문자 ID에 해당하는 문자를 반환, 정답

[ 3  0  2  0  0 11  5]
71+118 
_189 


### seq2seq 구현 

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [ ]:
class LSTM:
  def __init__(self,Wx,Wh,b):
    self.params = [Wx,Wh,b] # 가중치에는 4개분의 가중치가 담겨 있다. input gate, forget gate, output gate, block input의 가중치 4개
    self.grads = [np.zeros_like(Wx),np.zeros_like(Wh),np.zeros_like(b)] # 기울기를 저장하는 grads 인스턴스 변수 초기화
    self.cache = None       # 순전파시 중간 결과를 담을 cache변수로 역전파 계산에 사용하는 인스턴스 변수 cache를 초기화
  
  # 순전파
  def forward(self, x, h_prev, c_prev): # x는 입력 데이터, h_prev는 이전 시각의 은닉 상태, c_prev는 이전 시각의 기억 셀
    Wx, Wh, b = self.params
    N, H = h_prev.shape

    A = np.matmul(x, Wx) + np.matmul(h_prev, Wh) + b  # 4개의 가중치를 모아 아핀 변환을 수행하여 A를 계산

    f = A[:, :H]
    g = A[:, H:2*H]
    i = A[:, 2*H:3*H]
    o = A[:, 3*H:]

    f = sigmoid(f)                # forget gate
    g = np.tanh(g)                # 새로운 정보를 기억하기 위한 input
    i = sigmoid(i)                # input gate
    o = sigmoid(o)                # output gate

    c_next = f * c_prev + g * i   # 기억 셀(memory cell) 계산, c_next = f * c_prev + g * i
    h_next = o * np.tanh(c_next)  # 은닉 상태 계산, h_next = o * tanh(c_next)

    self.cache = (x, h_prev, c_prev, i, f, g, o, c_next) # cache에 순전파시 중간 결과(입력 데이터, 이전 시각의 은닉 상태, 이전 시각의 기억 셀, 4개의 gate, 기억 셀)를 저장
    return h_next, c_next                                # 은닉 상태(hidden state)와 기억 셀(memory cell)을 반환
  
  # 역전파
  def backward(self, dh_next, dc_next): # 상류에서 전해지는 기울기 dh_next와 dc_next를 받아 하류로 기울기를 전달
    Wx, Wh, b = self.params
    x, h_prev, c_prev, i, f, g, o, c_next = self.cache

    tanh_c_next = np.tanh(c_next)

    ds = dc_next + (dh_next * o) * (1 - tanh_c_next ** 2)  # ds는 tanh(c_next)의 역전파에서 내려온 기울기와 상류에서 전해지는 기울기 dh_next * o를 더한 값

    dc_prev = ds * f # 이전 시각 기억 셀(c_prev)의 기울기 dc_prev 계산, ds에 forget gate 출력 결과인 f를 곱함

    di = ds * g                 # di는 상류에서 전해지는 기울기 ds와 block input g를 곱한 값
    df = ds * c_prev            # df는 상류에서 전해지는 기울기 ds와 이전 시각 기억 셀 c_prev를 곱한 값
    do = dh_next * tanh_c_next  # do는 상류에서 전해지는 기울기 dh_next와 tanh(c_next)를 곱한 값
    dg = ds * i                 # dg는 상류에서 전해지는 기울기 ds와 input gate i를 곱한 값

    di *= i * (1 - i)           # 시그모이드 함수 기울기를 di에 곱하고 하류로 전달
    df *= f * (1 - f)           # 시그모이드 함수 기울기를 df에 곱하고 하류로 전달
    do *= o * (1 - o)           # 시그모이드 함수 기울기를 do에 곱하고 하류로 전달
    dg *= (1 - g ** 2)          # tanh 함수 기울기를 dg에 곱하고 하류로 전달

    dA = np.hstack((df, dg, di, do)) # 4개의 gate에 대한 기울기를 열 방향으로 합치기

    dWh = np.dot(h_prev.T, dA)  # dA와 h_prev의 내적을 구해 Wh에 대한 기울기 dWh를 구함
    dWx = np.dot(x.T, dA)       # dA와 x의 내적을 구해 Wx에 대한 기울기 dWx를 구함
    db = dA.sum(axis=0)         # dA의 각 행을 다 더해 db를 구함
    
    # dWx, dWh, db를 각각 grads의 0, 1, 2번째 인덱스에 저장, 4개(input gate, forget gate, output gate, block input)의 가중치에 대한 기울기를 저장
    self.grads[0][...] = dWx        
    self.grads[1][...] = dWh
    self.grads[2][...] = db

    dx = np.dot(dA, Wx.T)       # dx는 dA와 Wx의 내적을 구해 구함
    dh_prev = np.dot(dA, Wh.T)  # dh_prev는 dA와 Wh의 내적을 구해 구함

    return dx, dh_prev, dc_prev

In [ ]:
# T개 분의 시계열 데이터를 한꺼번에 처리하는 TimeLSTM 계층
class TimeLSTM:
  def __init__(self, Wx, Wh, b , stateful=False):
    self.params = [Wx, Wh, b]
    self.grads = [np.zeros_like(Wx), np.zeros_like(Wh), np.zeros_like(b)]
    self.layers = None

    self.h, self.c = None, None   # hidden state, cell state
    self.dh = None
    self.stateful = stateful
  
  def forward(self, xs):
    Wx, Wh, b = self.params
    N, T, D = xs.shape
    H = Wh.shape[0]

    self.layers = []
    hs = np.empty((N, T, H), dtype='f')     # 출력값을 저장할 변수 hs를 초기화

    if not self.stateful or self.h is None:
      self.h = np.zeros((N, H), dtype='f')
    if not self.stateful or self.c is None:
      self.c = np.zeros((N, H), dtype='f')
    
    for t in range(T):
      layer = LSTM(*self.params)
      self.h, self.c = layer.forward(xs[:, t, :], self.h, self.c)
      hs[:, t, :] = self.h

      self.layers.append(layer)
    
    return hs
  
  def backward(self, dhs):
    Wx, Wh, b = self.params
    N, T, H = dhs.shape
    D = Wx.shape[0]

    dxs = np.empty((N, T, D), dtype='f')
    dh, dc = 0, 0

    grads = [0, 0, 0]
    for t in reversed(range(T)):
      layer = self.layers[t]
      dx, dh, dc = layer.backward(dhs[:, t, :] + dh, dc)
      dxs[:, t, :] = dx
      for i, grad in enumerate(layer.grads):
        grads[i] += grad
    
    for i, grad in enumerate(grads):
      self.grads[i][...] = grad
    self.dh = dh
    return dxs
  
  def set_state(self, h, c=None):
    self.h, self.c = h, c
  
  # 상태 초기화
  def reset_state(self):
    self.h, self.c = None, None

In [ ]:
# encoder
class Encoder:
  def __init__(self, voab_size, wordvec_size, hidden_size): # vocab_size는 어휘 수, wordvec_size는 단어 벡터의 차원 수, hidden_size는 LSTM 계층의 은닉 상태 벡터의 차원 수
    V, D, H = voab_size, wordvec_size, hidden_size
    rn = np.random.randn # 랜덤값 생성

    # 가중치 초기화
    embed_W = (rn(V, D) / 100).astype('f')
    lstm_Wx = (rn(D, 4 * H) / np.sqrt(D)).astype('f')
    lstm_Wh = (rn(H, 4 * H) / np.sqrt(H)).astype('f')
    lstm_b = np.zeros(4 * H).astype('f')

    self.embed = TimeEmbedding(embed_W)
    self.lstm = TimeLSTM(lstm_Wx, lstm_Wh, lstm_b, stateful=False)

    self.params = self.embed.params + self.lstm.params
    self.grads = self.embed.grads + self.lstm.grads
    self.hs = None